# RUN_SUITE — reference regression

**Previous:** [00_OVERVIEW](00_OVERVIEW.ipynb)  
**Kernel:** Connect to Existing with Qwen already loaded (e.g. `119f847c`).

This notebook **reuses** `model`/`tok` in the kernel. It will error if they are missing (no second `from_pretrained`).


## 1. Path + live model check


In [ ]:
# Prefer REUSE — never force a fresh GPU load for the suite
import sys
from pathlib import Path

_here = Path.cwd().resolve()
for _p in [_here, *_here.parents]:
    if (_p / "_lib" / "cxr_boot.py").is_file():
        sys.path.insert(0, str(_p / "_lib"))
        break
    if (_p / "notebooks" / "_lib" / "cxr_boot.py").is_file():
        sys.path.insert(0, str(_p / "notebooks" / "_lib"))
        break
else:
    raise RuntimeError("Cannot find notebooks/_lib")

import cxr_boot
import cxr_suite

# setup(load_model=True) still REUSES if model/tok already in ns
ctx = cxr_boot.setup(layer=20, max_new=24, load_model=True)
model, tok = cxr_suite.require_live_model()
# refresh ctx fields if needed
NOTE = ctx["NOTE"]
LAYER = ctx["LAYER"]
MAX_NEW = ctx["MAX_NEW"]
PROMPT_A, PROMPT_B, PROMPT_TEST = ctx["PROMPT_A"], ctx["PROMPT_B"], ctx["PROMPT_TEST"]
suite_ctx = {
    "NOTE": NOTE,
    "LAYER": LAYER,
    "MAX_NEW": MAX_NEW,
    "PROMPT_A": PROMPT_A,
    "PROMPT_B": PROMPT_B,
    "PROMPT_TEST": PROMPT_TEST,
}
print("pid", __import__("os").getpid())
print("NOTE:", NOTE)
print("registry experiments:", [e["id"] for e in cxr_suite.load_registry()["experiments"]])


## 2. Run all enabled experiments


In [ ]:
# Full phase-1 suite (E00 smoke + sep sweep + causal grid). ~1–2 min on resident Qwen.
results = cxr_suite.run_tagged(model, tok, suite_ctx)
for r in results:
    print(r["id"], "→", "PASS" if r["pass"] else "FAIL", r["fails"][:3] if r["fails"] else "")


## 3. Optional — smoke tags only


In [ ]:
# Faster subset
# results_smoke = cxr_suite.run_tagged(model, tok, suite_ctx, tags=["smoke"])


## 4. Claim boundary

✓ Suite compared live metrics to `expected/*.json` and wrote `artifacts/experiments/`.  
✗ Passing does not expand Dual / `ground()` / M2.  
✗ Streamlit and Jupyter still cannot share one Python model object without a shared server.
